In [ ]:
import cv2
import csv
import math 
from ultralytics import YOLO
from collections import defaultdict
import numpy as np
import json
import random
import shutil
from pathlib import Path
import torch
import easyocr

### EasyOCR

In [ ]:

#Se cargan los modelos para detectar objetos y matrículas
model = YOLO('yolo11n.pt')  # Personas, bicicletas y coches
model_matriculas = YOLO("../../matriculas/runs/train_val_matriculas/weights/best.pt")

#Se inicializa EasyOCR para reconocer el texto de las matrículas
reader = easyocr.Reader(['es'], gpu=True)

#Se definen las etiquetas de las distintas clases
classNames = ["person", "bicycle", "car"]

#Se abre el vídeo que se va a procesar
vid = cv2.VideoCapture("../P4/C0142.mp4")

track_history = defaultdict(lambda: [])

#Se obtienen las características del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

#Se crea el vídeo de salida para almacenar los fotogramas procesados
out = cv2.VideoWriter("C0142_procesado_Ocr.mp4", fourcc, fps, (width, height))

#Se crea el archivo CSV y se definen los campos para almacenar los resultados
archivo_csv = open("C0142_resultados_Ocr.csv", "w", newline="", encoding="utf-8")
campos = [
    "fotograma", "tipo_objeto", "confianza",
    "identificador_tracking", "x1", "y1", "x2", "y2",
    "matricula_en_su_caso", "confianza_matricula",
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

escritor = csv.DictWriter(archivo_csv, fieldnames=campos)
escritor.writeheader()

#Se inicializan los contadores y los conjuntos para almacenar los IDs contabilizados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

#Se inicializa el contador de fotogramas
num_fotograma = 0

while(True):

    #Se obtiene cada fotograma del vídeo
    ret, img = vid.read()

    #Se comprueba que el fotograma se haya leído correctamente
    if ret:

        num_fotograma += 1

        #Se realiza el seguimiento de personas, bicicletas y coches
        results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

        #Se detectan las matrículas presentes en el fotograma
        results_matriculas = model_matriculas.predict(img, conf=0.25, verbose=False)

        #Se inicializa la lista para almacenar las matrículas detectadas
        matriculas = []

        for r_mat in results_matriculas:
            for box_mat in r_mat.boxes:

                #Se obtienen las coordenadas de cada matrícula detectada
                mx1, my1, mx2, my2 = box_mat.xyxy[0].cpu().numpy()
                mx1, my1, mx2, my2 = int(mx1), int(my1), int(mx2), int(my2)

                #Se ajustan las coordenadas para evitar que salgan de los límites de la imagen
                mx1 = max(0, mx1)
                my1 = max(0, my1)
                mx2 = min(width, mx2)
                my2 = min(height, my2)

                #Se obtiene la confianza de la detección de la matrícula
                confianza_mat = float(box_mat.conf[0])

                #Se recorta la región de la imagen correspondiente a la matrícula
                roi_mat = img[my1:my2, mx1:mx2]

                #Se inicializan las variables para almacenar el texto y la confianza del reconocimiento
                texto_mat = ""
                confianza_ocr = 0.0

                #Se comprueba que la región recortada contiene información
                if roi_mat.size > 0:

                    #Se convierte la imagen a escala de grises
                    roi_mat = cv2.cvtColor(roi_mat, cv2.COLOR_BGR2GRAY)

                    #Se amplía la imagen 5 veces mediante interpolación cúbica
                    roi_mat = cv2.resize(
                        roi_mat, None, fx=5, fy=5,
                        interpolation=cv2.INTER_CUBIC
                    )

                    #Se mejora el contraste de la imagen mediante CLAHE
                    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                    roi_mat = clahe.apply(roi_mat)

                    resultado_ocr = reader.readtext(roi_mat, allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')

                    #Se selecciona el resultado con mayor confianza de reconocimiento
                    if len(resultado_ocr) > 0:
                        mejor_resultado = max(resultado_ocr, key=lambda x: x[2])
                        texto_mat = mejor_resultado[1].replace(" ", "").upper()
                        confianza_ocr = float(mejor_resultado[2])

                #Se almacena la información obtenida de la matrícula
                matriculas.append({
                    "x1": mx1, "y1": my1, "x2": mx2, "y2": my2,
                    "confianza": confianza_mat,
                    "confianza_ocr": confianza_ocr,
                    "texto": texto_mat
                })

        #Se procesan las detecciones de personas, bicicletas y coches
        for r in results:

            boxes = r.boxes

            for box in boxes:

                #Se obtienen las coordenadas de la caja delimitadora de cada objeto
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                #Se ajustan las coordenadas para evitar que salgan de los límites de la imagen
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                #Se obtiene la confianza de la detección
                confidence = float(box.conf[0])
                print("Confianza =", round(confidence, 2))

                #Se identifica la clase del objeto detectado
                cls = int(box.cls[0])
                nombre_clase = classNames[cls]
                print("Clase =", nombre_clase)

                #Se obtiene el identificador de seguimiento del objeto
                if box.id is not None:
                    track_id = str(int(box.id[0]))
                else:
                    track_id = ""

                #Se inicializa la variable para almacenar la matrícula asociada
                matricula_asociada = None

                #Se buscan matrículas asociadas únicamente a los coches
                if cls == 2:

                    for mat in matriculas:

                        #Se calcula el centro de la caja delimitadora de la matrícula
                        centro_x = (mat["x1"] + mat["x2"]) // 2
                        centro_y = (mat["y1"] + mat["y2"]) // 2

                        #Se comprueba si el centro de la matrícula se encuentra dentro del coche
                        if (x1 <= centro_x <= x2 and
                            y1 <= centro_y <= y2):

                            #Se selecciona la matrícula con mayor confianza de detección
                            if (matricula_asociada is None or
                                mat["confianza"] > matricula_asociada["confianza"]):
                                matricula_asociada = mat

                #Se actualizan los contadores si el ID todavía no ha sido contabilizado
                if box.id is not None:

                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                #Se obtiene la información de la matrícula para almacenarla en el CSV
                if matricula_asociada is not None:
                    tiene_matricula = "Si"
                    confianza_matricula = matricula_asociada["confianza"]
                    mx1 = matricula_asociada["x1"]
                    my1 = matricula_asociada["y1"]
                    mx2 = matricula_asociada["x2"]
                    my2 = matricula_asociada["y2"]
                    texto_matricula = matricula_asociada["texto"]

                else:
                    tiene_matricula = "No"
                    confianza_matricula = ""
                    mx1 = ""
                    my1 = ""
                    mx2 = ""
                    my2 = ""
                    texto_matricula = ""

                #Se escribe la información de cada objeto detectado en el archivo CSV
                escritor.writerow({
                    "fotograma": num_fotograma,
                    "tipo_objeto": nombre_clase,
                    "confianza": confidence,
                    "identificador_tracking": track_id,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "matricula_en_su_caso": tiene_matricula,
                    "confianza_matricula": confianza_matricula,
                    "mx1": mx1,
                    "my1": my1,
                    "mx2": mx2,
                    "my2": my2,
                    "texto_matricula": texto_matricula
                })

                #Se asigna un color diferente a cada clase de objeto
                if cls == 0:
                    R = 128
                    G = 0
                    B = 0
                elif cls == 1:
                    R = 0
                    G = 128
                    B = 0
                elif cls == 2:
                    R = 0
                    G = 0
                    B = 128

                #Se dibujan la caja delimitadora y la etiqueta de seguimiento del objeto
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)

                cv2.putText(img, track_id + ' ' + nombre_clase, [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

                #Se muestra la información de la matrícula cuando se encuentra asociada a un coche
                if matricula_asociada is not None:

                    texto = texto_matricula

                    #Se indica cuando no se ha podido reconocer el texto de la matrícula
                    if texto == "":
                        texto = "Matricula no reconocida"

                    #Se dibujan la caja delimitadora y el texto reconocido de la matrícula
                    cv2.rectangle(img, (mx1, my1), (mx2, my2), (0, 255, 255), 2)

                    cv2.putText(img, texto, (mx1, max(20, my1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        #Se muestran los contadores de personas, bicicletas y coches en el fotograma
        cv2.putText(img, f'Personas: {contadores["person"]}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(img, f'Coches: {contadores["car"]}',
                    (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        #Se guarda el fotograma procesado en el vídeo de salida
        out.write(img)

        #Se muestra el fotograma procesado
        cv2.imshow('Vid', img)

    else:
        break

    #Se detiene la ejecución al pulsar ESC
    if cv2.waitKey(20) == 27:
        break

#Se libera el objeto de captura del vídeo
vid.release()

#Se libera el objeto de escritura del vídeo
out.release()

archivo_csv.close()

cv2.destroyAllWindows()

Confianza = 0.84
Clase = car
Confianza = 0.81
Clase = car
Confianza = 0.82
Clase = car
Confianza = 0.84
Clase = car
Confianza = 0.82
Clase = car
Confianza = 0.81
Clase = car
Confianza = 0.8
Clase = car
Confianza = 0.79
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.81
Clase = car
Confianza = 0.8
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.79
Clase = car
Confianza = 0.79
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.77
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.77
Clase = car
Confianza = 0.78
Clase = car
Confianza = 0.75
Clase = car
Confianza = 0.73
Clase = car
Confianza = 0.74
Clase = car
Confianza = 0.72
Clase = car
Confianza = 0.7
Clase = car
Confianza = 0.71
Clase = car
Confianza = 0.69
Clase = car
Confianza = 0.67
Clase = car
Confianza = 0.65
Clase = car
Confianza = 0.63
Clase = car
Confianza = 0.62
Clase = car
Confianza = 0.64
Clase = car
Confianza = 0.64
Clase = car
Confianza = 0.63
Clase = car
Confianza = 0.62


In [ ]:
# RECORTE DE MATRICULAS A PARTIR DE ANOTACIONES JSON
import cv2
import csv
import json
import numpy as np
from pathlib import Path

# localizar la raiz del proyecto
carpeta_actual = Path.cwd()

carpeta_proyecto = next(
    (p for p in [carpeta_actual, *carpeta_actual.parents]
     if (p / "matriculas" / "images").exists()
     and (p / "matriculas" / "annotations").exists()),
    None
)

if carpeta_proyecto is None:
    raise FileNotFoundError("No se encuentra la carpeta matriculas")

# rutas
carpeta_imagenes = carpeta_proyecto / "matriculas" / "images"
carpeta_anotaciones = carpeta_proyecto / "matriculas" / "annotations"
carpeta_p4b = carpeta_proyecto / "P4b"
carpeta_recortes = carpeta_p4b / "images_recortadas"

carpeta_recortes.mkdir(parents=True, exist_ok=True)

# buscar imagenes
extensiones = [".jpg", ".jpeg", ".png", ".bmp"]
imagenes = [
    p for p in carpeta_imagenes.iterdir()
    if p.suffix.lower() in extensiones
]

resultados = []
total_sin_json = 0
total_sin_matricula = 0
total_error_imagen = 0

# margen alrededor de la matricula
margen = 0.08

# procesar imagenes
for imagen_path in imagenes:

    archivo_json = carpeta_anotaciones / (imagen_path.stem + ".json")

    if not archivo_json.exists():
        total_sin_json += 1
        continue

    # cargar imagen compatible con rutas Unicode
    img = cv2.imdecode(
        np.fromfile(str(imagen_path), dtype=np.uint8),
        cv2.IMREAD_COLOR
    )

    if img is None:
        total_error_imagen += 1
        continue

    # cargar anotaciones
    with open(archivo_json, "r", encoding="utf-8") as f:
        anotaciones = json.load(f)

    alto, ancho = img.shape[:2]

    for i, anotacion in enumerate(anotaciones):

        matricula = str(anotacion.get("plate", "")).strip().upper()

        if not matricula or matricula == "NONE":
            total_sin_matricula += 1
            continue

        # coordenadas normalizadas
        cx, cy, w, h = anotacion["rectangle"]

        # convertir a pixeles y añadir margen
        x1 = int((cx - w / 2 - w * margen) * ancho)
        y1 = int((cy - h / 2 - h * margen) * alto)
        x2 = int((cx + w / 2 + w * margen) * ancho)
        y2 = int((cy + h / 2 + h * margen) * alto)

        # ajustar a los limites de la imagen
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(ancho, x2)
        y2 = min(alto, y2)

        # recortar matricula
        recorte = img[y1:y2, x1:x2]

        if recorte.size == 0:
            continue

        # guardar recorte
        nombre = f"{imagen_path.stem}_{i}.jpg"
        ruta_salida = carpeta_recortes / nombre

        # guardar recorte compatible con rutas Unicode
        correcto, buffer = cv2.imencode(".jpg", recorte)

        if correcto:
            buffer.tofile(str(ruta_salida))

            if ruta_salida.exists():
                resultados.append({
                    "imagen": nombre,
                    "matricula_real": matricula,
                    "imagen_original": imagen_path.name
                })

# guardar CSV
ruta_csv = carpeta_p4b / "matriculas_reales.csv"

with open(ruta_csv, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(
        f,
        fieldnames=["imagen", "matricula_real", "imagen_original"]
    )
    escritor.writeheader()
    escritor.writerows(resultados)

# resumen
print("DATASET DE MATRICULAS PREPARADO")
print("--------------------------------")
print("Imagenes originales:", len(imagenes))
print("Recortes generados:", len(resultados))
print("Imagenes sin JSON:", total_sin_json)
print("Anotaciones sin matricula:", total_sin_matricula)
print("Errores al cargar imagenes:", total_error_imagen)
print("--------------------------------")
print("Carpeta:", carpeta_recortes)
print("CSV:", ruta_csv)


DATASET DE MATRICULAS PREPARADO
--------------------------------
Imagenes originales: 2387
Recortes generados: 3241
Imagenes sin JSON: 0
Anotaciones sin matricula: 1994
Errores al cargar imagenes: 0
--------------------------------
Carpeta: f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4b\images_recortadas
CSV: f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4b\matriculas_reales.csv


In [17]:

import csv
import cv2
import numpy as np
import torch
import easyocr
import time
from pathlib import Path

# rutas
carpeta_recortes = Path.cwd() / "images_recortadas"
ruta_csv = Path.cwd() / "matriculas_reales.csv"

# cargar etiquetas reales
with open(ruta_csv, "r", encoding="utf-8") as f:
    datos = list(csv.DictReader(f))

# seleccionar solamente las imagenes que existen
datos_disponibles = [
    fila for fila in datos
    if (carpeta_recortes / fila["imagen"]).is_file()
]

# cargar EasyOCR
usar_gpu = torch.cuda.is_available()
reader = easyocr.Reader(["en"], gpu=usar_gpu)

print("GPU disponible:", usar_gpu)
print("Imagenes en el CSV:", len(datos))
print("Imagenes disponibles:", len(datos_disponibles))
print("--------------------------------")

resultados = []
tiempo_total = 0

# procesar imagenes disponibles
for i, fila in enumerate(datos_disponibles):

    ruta = carpeta_recortes / fila["imagen"]

    # cargar imagen
    imagen = cv2.imdecode(
        np.fromfile(str(ruta), dtype=np.uint8),
        cv2.IMREAD_COLOR
    )

    if imagen is None:
        print("No se pudo cargar:", ruta.name)
        continue

    # reconocer matricula y medir tiempo
    inicio = time.time()

    lectura = reader.readtext(
        imagen,
        allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
        detail=1
    )

    tiempo = time.time() - inicio
    tiempo_total += tiempo

    # unir textos detectados
    prediccion = "".join(r[1] for r in lectura)
    real = fila["matricula_real"].upper()

    resultados.append({
        "imagen": fila["imagen"],
        "real": real,
        "prediccion": prediccion,
        "correcta": prediccion == real,
        "tiempo": tiempo
    })

    print(f"{i + 1}/{len(datos_disponibles)} | Real: {real} | OCR: {prediccion}")

# calcular resultados
total = len(resultados)
correctas = sum(r["correcta"] for r in resultados)

if total > 0:
    print("\nRESULTADOS DE EASYOCR")
    print("--------------------------------")
    print("Imagenes procesadas:", total)
    print("Lecturas exactas:", correctas)
    print("Lecturas incorrectas:", total - correctas)
    print("Exactitud:", round(correctas / total * 100, 2), "%")
    print("Tiempo total:", round(tiempo_total, 2), "segundos")
    print("Tiempo medio:", round(tiempo_total / total * 1000, 2), "ms")

# guardar predicciones
ruta_resultados = Path.cwd() / "resultados_easyocr.csv"

with open(ruta_resultados, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(
        f,
        fieldnames=["imagen", "real", "prediccion", "correcta", "tiempo"]
    )
    escritor.writeheader()
    escritor.writerows(resultados)

print("\nResultados guardados en:", ruta_resultados)


GPU disponible: True
Imagenes en el CSV: 3241
Imagenes disponibles: 975
--------------------------------
1/975 | Real: 8712CHC | OCR: 8712CHC
2/975 | Real: 8275GMN | OCR: 8275GIN
3/975 | Real: 8859HGP | OCR: 8859HGP
4/975 | Real: 9195HSV | OCR: 19435HSU
5/975 | Real: 0173FVB | OCR: FVB00173
6/975 | Real: 0224CXR | OCR: 022LCXR
7/975 | Real: 0270DRN | OCR: 02705DRNREMVDTMTURISFLA9T05
8/975 | Real: 4050FNM | OCR: LO5OFMM
9/975 | Real: 0402HCF | OCR: 10402HCF
10/975 | Real: GC0649BY | OCR: GCO649BY
11/975 | Real: 0617KFZ | OCR: 0617KFZ
12/975 | Real: 8697HXR | OCR: 
13/975 | Real: 0696CMB | OCR: CHB006964
14/975 | Real: 0848GLN | OCR: 0848GLN
15/975 | Real: 0951MVN | OCR: E0951MVN
16/975 | Real: 6324KZP | OCR: 
17/975 | Real: 4610FNP | OCR: 1L6T0FHP
18/975 | Real: 4721LJT | OCR: L12IJL
19/975 | Real: 4721LJT | OCR: LZLJ
20/975 | Real: 4170KDX | OCR: KDX4170
21/975 | Real: 2062MTM | OCR: 
22/975 | Real: 8838LKV | OCR: 780304V
23/975 | Real: 2783LXF | OCR: 2283LXE
24/975 | Real: 1864GKM | O

## Resultados de EasyOCR

### 1. Configuración de la evaluación

Se ha utilizado EasyOCR para realizar el reconocimiento de matrículas a partir de los recortes generados previamente.

- **Modelo:** EasyOCR preentrenado.
- **Imágenes evaluadas:** 975.
- **Datos:** recortes de matrículas con su correspondiente etiqueta real.
- **Tipo de evaluación:** inferencia con un modelo preentrenado, sin entrenamiento adicional.

### 2. Resultados obtenidos

| Métrica | Resultado |
|---|---:|
| Imágenes procesadas | 975 |
| Predicciones correctas | 259 |
| Predicciones incorrectas | 716 |
| Predicciones vacías | 118 |
| Exact Match | **26,56 %** |
| CER | **45,77 %** |
| Tiempo total | 30,78 s |
| Tiempo medio por imagen | 31,57 ms |
| Tiempo mediano por imagen | 24,94 ms |

### 3. Interpretación de los resultados

EasyOCR ha conseguido reconocer correctamente 259 de las 975 matrículas, alcanzando un Exact Match del 26,56 %.

El CER obtenido es del 45,77 %, lo que indica que existe una proporción considerable de caracteres incorrectamente reconocidos respecto al texto real.

Por otro lado, el tiempo de procesamiento es bastante reducido, con una media de aproximadamente 31,57 milisegundos por imagen. Esto permite realizar el reconocimiento de forma rápida.

In [21]:
# Evaluación de reconocimiento de matrículas con SmolVLM
import csv
import time
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

# cargar modelo
modelo = "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(modelo)

model = AutoModelForVision2Seq.from_pretrained(
    modelo,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

# rutas
carpeta = Path.cwd() / "images_recortadas"
archivo_csv = Path.cwd() / "matriculas_reales.csv"

# cargar matriculas reales
with open(archivo_csv, encoding="utf-8-sig") as f:
    datos = list(csv.DictReader(f))

# quedarse solo con las imagenes disponibles
datos = [d for d in datos if (carpeta / d["imagen"]).is_file()]

resultados = []

# prompt para leer la matricula
mensajes = [{
    "role": "user",
    "content": [
        {"type": "image"},
        {"type": "text", "text": "Read the license plate. Return only the letters and numbers."}
    ]
}]

texto = processor.apply_chat_template(
    mensajes,
    add_generation_prompt=True
)

# recorrer imagenes
for i, fila in enumerate(datos):

    imagen = Image.open(carpeta / fila["imagen"]).convert("RGB")

    entradas = processor(
        text=texto,
        images=[imagen],
        return_tensors="pt"
    ).to(device)

    inicio = time.time()

    with torch.inference_mode():
        salida = model.generate(**entradas, max_new_tokens=20)

    tiempo = time.time() - inicio

    # obtener matricula detectada
    tokens = salida[:, entradas["input_ids"].shape[1]:]
    prediccion = processor.batch_decode(
        tokens,
        skip_special_tokens=True
    )[0].strip()

    resultados.append({
        "imagen": fila["imagen"],
        "matricula_real": fila["matricula_real"],
        "prediccion": prediccion,
        "tiempo": round(tiempo, 4)
    })

    imagen.close()

    if (i + 1) % 25 == 0:
        print(f"Procesadas {i + 1}/{len(datos)}")

# guardar resultados
with open("resultados_smolvlm.csv", "w", encoding="utf-8-sig", newline="") as f:
    escritor = csv.DictWriter(f, fieldnames=resultados[0].keys())
    escritor.writeheader()
    escritor.writerows(resultados)

print("\nEvaluacion terminada.")
print("Imagenes procesadas:", len(resultados))
print("Resultados guardados en resultados_smolvlm.csv")


Procesadas 25/975
Procesadas 50/975
Procesadas 75/975
Procesadas 100/975
Procesadas 125/975
Procesadas 150/975
Procesadas 175/975
Procesadas 200/975
Procesadas 225/975
Procesadas 250/975
Procesadas 275/975
Procesadas 300/975
Procesadas 325/975
Procesadas 350/975
Procesadas 375/975
Procesadas 400/975
Procesadas 425/975
Procesadas 450/975
Procesadas 475/975
Procesadas 500/975
Procesadas 525/975
Procesadas 550/975
Procesadas 575/975
Procesadas 600/975
Procesadas 625/975
Procesadas 650/975
Procesadas 675/975
Procesadas 700/975
Procesadas 725/975
Procesadas 750/975
Procesadas 775/975
Procesadas 800/975
Procesadas 825/975
Procesadas 850/975
Procesadas 875/975
Procesadas 900/975
Procesadas 925/975
Procesadas 950/975
Procesadas 975/975

Evaluacion terminada.
Imagenes procesadas: 975
Resultados guardados en resultados_smolvlm.csv


hay que instalar "transformers" en anaconda

## Resultados de SmolVLM

### 1. Configuración de la evaluación

Se ha utilizado SmolVLM para realizar el reconocimiento de matrículas a partir de los mismos recortes empleados en la evaluación de EasyOCR.

- **Modelo:** `HuggingFaceTB/SmolVLM-256M-Instruct`.
- **Parámetros:** 256 millones.
- **Imágenes evaluadas:** 975.
- **Datos:** recortes de matrículas con su correspondiente etiqueta real.
- **Tipo de evaluación:** inferencia con un modelo preentrenado, sin entrenamiento adicional.

### 2. Resultados obtenidos

| Métrica | Resultado |
|---|---:|
| Imágenes procesadas | 975 |
| Predicciones correctas | 587 |
| Predicciones incorrectas | 388 |
| Predicciones vacías | 0 |
| Exact Match | **60,21 %** |
| CER | **15,58 %** |
| Tiempo total | 27 min 18 s |
| Tiempo medio por imagen | 1,68 s |
| Tiempo mínimo por imagen | 0,64 s |
| Tiempo máximo por imagen | 3,95 s |

### 3. Interpretación de los resultados

SmolVLM ha conseguido reconocer correctamente 587 de las 975 matrículas, alcanzando un Exact Match del 60,21 %.

El CER obtenido es del 15,58 %, lo que indica que la mayoría de los caracteres reconocidos coinciden con los reales, aunque no siempre se consigue identificar la matrícula completa.

Por otro lado, el tiempo de procesamiento es considerablemente elevado, con una media de aproximadamente 1,68 segundos por imagen. Esto supone un coste computacional importante si se pretende utilizar el modelo en aplicaciones de reconocimiento en tiempo real.

### 4. Conclusiones

SmolVLM obtiene resultados de reconocimiento considerablemente superiores a los de EasyOCR en el conjunto evaluado.

Su porcentaje de matrículas completamente reconocidas alcanza el 60,21 %, mientras que el CER se reduce hasta el 15,58 %.

Sin embargo, esta mejora en precisión tiene un coste importante en tiempo de ejecución, ya que necesita aproximadamente 27 minutos y 18 segundos para procesar las 975 imágenes.

Por tanto, SmolVLM resulta más interesante cuando se prioriza la precisión del reconocimiento, mientras que su elevado tiempo de inferencia puede limitar su utilización en aplicaciones que necesiten procesar imágenes en tiempo real.

## Comparación entre EasyOCR y SmolVLM

Se han comparado ambos modelos utilizando exactamente el mismo conjunto de 975 recortes de matrículas, con el objetivo de evaluar sus diferencias en precisión y velocidad de procesamiento.

### 1. Comparación de resultados

| Métrica | EasyOCR | SmolVLM |
|---|---:|---:|
| Imágenes evaluadas | 975 | 975 |
| Predicciones correctas | 259 | 587 |
| Predicciones incorrectas | 716 | 388 |
| Predicciones vacías | 118 | 0 |
| Exact Match | 26,56 % | **60,21 %** |
| CER | 45,77 % | **15,58 %** |
| Tiempo total | **30,78 s** | 27 min 18 s |
| Tiempo medio | **31,57 ms** | 1,68 s |

### 2. Análisis comparativo

Los resultados muestran que SmolVLM obtiene una precisión considerablemente superior a EasyOCR.

En concreto, SmolVLM mejora el Exact Match en aproximadamente 33,65 puntos porcentuales y reduce el CER en 30,19 puntos porcentuales.

Sin embargo, esta mejora tiene un coste importante en tiempo de ejecución. EasyOCR procesa las 975 imágenes en aproximadamente 31 segundos, mientras que SmolVLM necesita más de 27 minutos.

Por tanto, SmolVLM tarda aproximadamente 53 veces más en procesar el mismo conjunto de imágenes.

### 3. Conclusiones

La comparación permite observar que existe una diferencia importante entre ambos modelos:

- **EasyOCR:** destaca por su elevada velocidad de procesamiento, aunque obtiene una precisión inferior en el reconocimiento de matrículas.
- **SmolVLM:** consigue resultados considerablemente mejores en precisión, pero presenta un tiempo de inferencia mucho más elevado.

En consecuencia, la elección del modelo depende de los requisitos de la aplicación. Si se prioriza la velocidad y el procesamiento en tiempo real, EasyOCR puede resultar más adecuado. En cambio, si se busca obtener un mayor porcentaje de matrículas correctamente reconocidas y se dispone de más tiempo de procesamiento, SmolVLM ofrece mejores resultados.

En definitiva, SmolVLM obtiene los mejores resultados de reconocimiento en este experimento, mientras que EasyOCR presenta una ventaja clara en velocidad.